---
technologies: "Data Analysis with Local-Cloud integration"
category: "Demonstration of technology"
difficulty: "Expert"
---

# Data Analysis with Local-Cloud Integration

## Used material

1. <span id="used-material-1"></span> [ICEbreaker package](https://github.com/K123AsJ0k1/ice-mlops-mlch/tree/main/applications/packages/icebreaker)

## How to data analysis with Local-Cloud integration?

Assuming you have done the dataset and model setup in the [setup for demonstration chapter](./12_setup_for_demonstration.ipynb), we can proceed with the demonstrations data analysis stage. The objectives of this stage are the following:

- Gather row and character statistics from internal and external datasets
- Extract features from the datasets
    - Give relevance grades for internal material for RAG ranking
    - Classify language and format for external material for dataset sampling

We will do this by first doing local data analysis on datasets under 7k rows and then use local-cloud infra to run a data analysis pipeline for remaining datasets:

- Local
    - Internal datasets
        - Tutorial parts 1-8 
    - External datasets (7288 rows):
        - openai-humaneval = 164 rows and 1 table
        - mbpp = 974 rows and 1 table
        - bash-command-data = 6150 rows and 1 table
- Local-cloud (4 652 668 rows):
    - opc-sft-stage2-educational_instruct = 118 278 rows and 6 table
    - opc-sft-stage2-mceval_instruct = 35 943 rows and 2 table
    - opc-sft-stage2-evol_instruct = 111 183 rows and 6 table
    - opc-sft-stage2-package_instruct = 170 943 rows and 9 table
    - opc-sft-stage1-filtered_infinity_instruct = 1 027 064 rows and 52 table
    - opc-sft-stage1-realuser_instruct = 675 837 rows and 34 table
    - opc-sft-stage1-largescale_diverse_instruct = 2 513 420 rows and 126 table

We will begin with setting up the enviroment:

In [1]:
from icebreaker.workflow.utility import workflow_setup_enviroment
from icebreaker.mlflow.setup import mlflow_setup_client

workflow_setup = workflow_setup_enviroment(
    env_path = '(fill)/.ssh/.env',
    swift_auth_url = 'https://pouta.csc.fi:5001/v3',
    swift_auth_version = '3',
    swift_pre_auth_url = 'https://a3s.fi:443/swift/v1'
)

workflow_mlflow_client = mlflow_setup_client(
    mlflow_parameters = {
        'tracking-uri': 'http://mlflow.oss:7001',
        's3-endpoint-url': 'http://localhost:7009',
        'aws-access-key-id': 'minioadmin',
        'aws-secret-access-key': 'minioadmin'
    }
)

/home/sfniila/main_staging/ice-mlops-mlch/tutorials/studying/part-7/part_7_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We can then do feature extraction on the internal material with the following:

In [ ]:
from icebreaker.workflow.use import workflow_feature_extract_internal_datasets
internal_dataset_extraction_total_time = workflow_feature_extract_internal_datasets(
    swift_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'experiment',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-serialization': 'pickle'
    },
    dataset_paths = [
        'DATA/SOURCE/internal-tutorial-v1-part-1.pkl',
        'DATA/SOURCE/internal-tutorial-v1-part-2.pkl',
        'DATA/SOURCE/internal-tutorial-v1-part-3.pkl',
        'DATA/SOURCE/internal-tutorial-v1-part-4.pkl',
        'DATA/SOURCE/internal-tutorial-v1-part-5.pkl',
        'DATA/SOURCE/internal-tutorial-v1-part-6.pkl',
        'DATA/SOURCE/internal-tutorial-v1-part-7.pkl',
        'DATA/SOURCE/internal-tutorial-v1-part-8.pkl'
    ],
    ref_column = 'ref-paths',
    path_column = 'absolute-path',
    material_grades = {
        'md': ('primary',3),
        'ipynb': ('primary',3),
        'py': ('secondary',1),
        'yaml': ('secondary',1),
        'txt': ('secondary',1),
        'sh': ('secondary',1),
        'env': ('secondary',1)
    },
    path_start_prefix = 'extraction',
    group_columns = [
        'part', 
        'document', 
        'chapter'
    ],
    value_column = 'idx',
    weigth_column = 'relevance'
)

We can do the data analysis for the internal material in the following way:

In [ ]:
from icebreaker.analysis.pipe import analysis_internal_pipe

formatted_internal_data = analysis_internal_pipe(
    swift_client = workflow_setup['swift-client'],
    mlflow_client = work_mlflow_client,
    experiment_name = 'mlch-code-assistant-rag-pipeline',
    run_name = 'internal-step-0-lt1-analysis',
    run_tags = {
        'workflow-type': 'local-cloud',
        'pipeline-step': 'internal-data-analysis'
    },
    run_parameters = {
        'used_enviroment': 'local-central-lt1',
        'utilization_cost_sec': 0.00001742,
        'utilization_cost_currency': 'EUR'
    },
    dataset_paths = [
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v1-part-1.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v1-part-2.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v1-part-3.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v1-part-4.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v1-part-5.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v1-part-6.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v1-part-7.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v1-part-8.pkl'
    ],
    storage_parameters = {
        'bucket-target': 'experiment',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-serialization': 'pickle'
    },
    name_start_prefix = 'extraction'
)

We can do the data analysis for the external material in the following way:

In [ ]:
from icebreaker.analysis.pipe import analysis_external_pipe

formatted_external_data = analysis_external_pipe(
    swift_client = workflow_setup['swift-client'],
    mlflow_client = work_mlflow_client,
    experiment_name = 'mlch-code-assistant-rag-pipeline',
    run_name = 'external-step-0-lt1-analysis',
    run_tags = {
        'workflow-type': 'local-cloud',
        'pipeline-step': 'external-data-analysis'
    },
    run_parameters = {
        'used_enviroment': 'local-central-lt1',
        'utilization_cost_sec': 0.00001742,
        'utilization_cost_currency': 'EUR'
    },
    dataset_paths = [
        'DATA/SOURCE/external-openai-humaneval.parquet',
        'DATA/SOURCE/external-mbpp.parquet',
        'DATA/SOURCE/external-bash-command-data.parquet'
    ],
    storage_parameters = {
        'bucket-target': 'experiment',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-serialization': 'parquet'
    },
    dataset_defaults = {
        'external-openai-humaneval': {
            'language-default': 'en',
            'format-default': 'python'
        },
        'external-mbpp': {
            'language-default': 'en',
            'format-default': 'python'
        },
        'external-bash-command-data': {
            'language-default': 'en',
            'format-default': 'bash'
        }
    },
    name_start_prefix = 'extraction'
)

With the local analysis done, we can setup a local-cloud data analysis pipeline using the [local parameters](./parameters/local-cluster-parameters.yaml), [cloud parameters](./parameters/cloud-cluster-parameters.yaml), [data analysis Ray script](./ray/data_analysis_external_data/main.py) and [pipeline details](./details/local_cloud_data_analysis_details.yaml) in the following way: 

In [19]:
from icebreaker.models.clusters import Clusters
from icebreaker.misc.yaml import get_validated_yamls
from icebreaker.misc.dict import update_nested_dict
from icebreaker.costs.use import costs_local_spending_per_second, costs_csc_cloud_spending_per_second
from icebreaker.objects.use import objects_nested_update
from icebreaker.ray.setup import ray_store_job
from icebreaker.setup.fragments import compose_yaml_dict
from icebreaker.setup.pipeline import create_pipeline_inputs
from icebreaker.models.pipeline_arguments import PipelineArguments

local_cloud_cluster_yamls = get_validated_yamls(
    relative_paths = [
        './parameters/local-cluster-parameters.yaml',
        './parameters/cloud-cluster-parameters.yaml'
    ],
    yaml_validator = Clusters,
    key_replacer = {
        'local-cluster-parameters': 'local',
        'cloud-cluster-parameters': 'cloud'
    }
)

controller_update_dict = {
    'local': {
        'clusters': {
            'lt1': {
                'activate': False
            },
            'lt2': {
                'activate': False
            },
            'lt3': {
                'activate': False
            }
        }
    },
    'cloud': {
        'clusters': {
            'vm2': {
                'activate': False
            }
        }
    }
}

pipeline_cluster_yaml = update_nested_dict(
    target_dict = local_cloud_cluster_yamls, 
    update_dict = controller_update_dict
)

cost_update_dict = {
    'local': {
        'clusters': {
            'lt1': {
                'utilization-cost-sec': costs_local_spending_per_second(
                    hardware_cost = local_cloud_cluster_yamls['local']['clusters']['lt1']['hardware-costs'],
                    expected_years = local_cloud_cluster_yamls['local']['clusters']['lifetime-expecation-year'],
                )
            },
            'lt2': {
                'utilization-cost-sec': costs_local_spending_per_second(
                    hardware_cost = local_cloud_cluster_yamls['local']['clusters']['lt2']['hardware-costs'],
                    expected_years = local_cloud_cluster_yamls['local']['clusters']['lifetime-expecation-year'],
                )
            },
            'lt3': {
                'utilization-cost-sec': costs_local_spending_per_second(
                    hardware_cost = local_cloud_cluster_yamls['local']['clusters']['lt3']['hardware-costs'],
                    expected_years = local_cloud_cluster_yamls['local']['clusters']['lifetime-expecation-year'],
                )
            }
        }
    },
    'cloud': {
        'clusters': {
            'vm1': {
                'utilization-cost-sec': costs_csc_cloud_spending_per_second(
                    package_cost_euros = local_cloud_cluster_yamls['cloud']['clusters']['package-cost-euros'],
                    package_billing_units = local_cloud_cluster_yamls['cloud']['clusters']['package-billing-units'],
                    spent_units_per_hour = local_cloud_cluster_yamls['cloud']['clusters']['vm1']['spent-units-per-hour']
                )
            },
            'vm2': {
                'utilization-cost-sec': costs_csc_cloud_spending_per_second(
                    package_cost_euros = local_cloud_cluster_yamls['cloud']['clusters']['package-cost-euros'],
                    package_billing_units = local_cloud_cluster_yamls['cloud']['clusters']['package-billing-units'],
                    spent_units_per_hour = local_cloud_cluster_yamls['cloud']['clusters']['vm2']['spent-units-per-hour']
                )
            }
        }
    }
}

pipeline_cluster_yaml = update_nested_dict(
    target_dict = pipeline_cluster_yaml, 
    update_dict = cost_update_dict
)


update_status = objects_nested_update(
    swift_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'pipeline',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-name': 'mana',
        'object-serialization': 'pickle',
        'path-replacers': {
            'name': 'local-cloud-cluster.pkl'
        },
        'path-names': [],
        'debug-prints': True,
        'lock-parameters': {},
        'lock-location': None,
        'overwrite': True
    },
    object_input = controller_update_dict
)

used_job_storing_time_2 = ray_store_job(
    storage_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'pipeline',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com'
    },
    ray_runtime = {
        'working_dir': '(fill)/ice-mlops-mlch/tutorials/studying/part-7/ray/data_analysis_external_data',
        'pip': '(fill)/ice-mlops-mlch/tutorials/studying/part-7/ray/data_analysis_external_data/requirements.txt'
    }
)

workflow_steps = [
    'step-1',
    'step-2',
    'step-3',
    'step-4',
    'step-5',
    'step-6',
    'step-7'
]

time_object_name_update = {
    'storage': 'fill',
    'processing': {
        'step-1': {
            'general': 'fill'
        },
        'step-2': {
            'general': 'fill'
        },
        'step-3': {
            'general': 'fill'
        },
        'step-4': {
            'general': 'fill'
        },
        'step-5': {
            'general': 'fill'
        },
        'step-6': {
            'general': 'fill'
        },
        'step-7': {
            'general': 'fill'
        }
    }
}

data_analysis_pipeline_arguments = compose_yaml_dict(
    fragments_folder = '(fill)/ice-mlops-mlch/tutorials/studying/part-7/fragments',
    fragments_prefix = '_fragment.yaml',
    root_fragment = 'parameters',
    details_folder = '(fill)/ice-mlops-mlch/tutorials/studying/part-7/details',
    details_prefix = '_analysis_details.yaml',
    env_dict = {},
    dict_inputs = {
        'run': 'local-cloud-data-analysis-run',
        'experiment': 'local-cloud-rag-assistant-experiment',
        'storage|swift': workflow_setup['swift-parameters'],
        'integration|cluster-yamls': pipeline_cluster_yaml,
        'integration|workflow-steps': workflow_steps,
        'processing|step-1|general|swift': workflow_setup['swift-parameters'],
        'processing|step-1|cluster|cloud-vm1|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-1|cluster|cloud-vm1|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-2|general|swift': workflow_setup['swift-parameters'],
        'processing|step-2|cluster|cloud-vm1|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-2|cluster|cloud-vm1|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-3|general|swift': workflow_setup['swift-parameters'],
        'processing|step-3|cluster|cloud-vm1|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-3|cluster|cloud-vm1|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-3|cluster|cloud-vm2|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-sec'],
        'processing|step-3|cluster|cloud-vm2|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-currency'],
        'processing|step-3|cluster|local-lt3|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-sec'],
        'processing|step-3|cluster|local-lt3|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-currency'],
        'processing|step-4|general|swift': workflow_setup['swift-parameters'],
        'processing|step-4|cluster|cloud-vm1|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-4|cluster|cloud-vm1|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-4|cluster|cloud-vm2|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-sec'],
        'processing|step-4|cluster|cloud-vm2|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-currency'],
        'processing|step-4|cluster|local-lt1|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt1']['utilization-cost-sec'],
        'processing|step-4|cluster|local-lt1|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt1']['utilization-cost-currency'],
        'processing|step-4|cluster|local-lt2|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-sec'],
        'processing|step-4|cluster|local-lt2|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-currency'],
        'processing|step-4|cluster|local-lt3|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-sec'],
        'processing|step-4|cluster|local-lt3|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-currency'],
        'processing|step-5|general|swift': workflow_setup['swift-parameters'],
        'processing|step-5|cluster|cloud-vm1|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-5|cluster|cloud-vm1|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-5|cluster|cloud-vm2|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-sec'],
        'processing|step-5|cluster|cloud-vm2|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-currency'],
        'processing|step-5|cluster|local-lt1|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt1']['utilization-cost-sec'],
        'processing|step-5|cluster|local-lt1|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt1']['utilization-cost-currency'],
        'processing|step-5|cluster|local-lt2|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-sec'],
        'processing|step-5|cluster|local-lt2|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-currency'],
        'processing|step-5|cluster|local-lt3|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-sec'],
        'processing|step-5|cluster|local-lt3|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-currency'],
        'processing|step-6|general|swift': workflow_setup['swift-parameters'],
        'processing|step-6|cluster|cloud-vm1|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-6|cluster|cloud-vm1|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-6|cluster|cloud-vm2|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-sec'],
        'processing|step-6|cluster|cloud-vm2|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-currency'],
        'processing|step-6|cluster|local-lt1|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt1']['utilization-cost-sec'],
        'processing|step-6|cluster|local-lt1|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt1']['utilization-cost-currency'],
        'processing|step-6|cluster|local-lt2|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-sec'],
        'processing|step-6|cluster|local-lt2|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-currency'],
        'processing|step-6|cluster|local-lt3|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-sec'],
        'processing|step-6|cluster|local-lt3|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-currency'],
        'processing|step-7|general|swift': workflow_setup['swift-parameters'],
        'processing|step-7|cluster|cloud-vm1|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-7|cluster|cloud-vm1|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-7|cluster|cloud-vm2|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-sec'],
        'processing|step-7|cluster|cloud-vm2|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm2']['utilization-cost-currency'],
        'processing|step-7|cluster|local-lt1|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt1']['utilization-cost-sec'],
        'processing|step-7|cluster|local-lt1|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt1']['utilization-cost-currency'],
        'processing|step-7|cluster|local-lt2|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-sec'],
        'processing|step-7|cluster|local-lt2|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-currency'],
        'processing|step-7|cluster|local-lt3|process|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-sec'],
        'processing|step-7|cluster|local-lt3|process|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-currency']
    }, 
    output_folder = '(fill)/ice-mlops-mlch/tutorials/studying/part-7/generated',
    output_name = 'data_analysis_pipeline_input'
)

pydantic_validation = PipelineArguments.model_validate(data_analysis_pipeline_arguments)
pipeline_run_name = data_analysis_pipeline_arguments['run']
pipeline_experiment_name = data_analysis_pipeline_arguments['experiment']
if 'run' in data_analysis_pipeline_arguments:
    del data_analysis_pipeline_arguments['run']
if 'experiment' in data_analysis_pipeline_arguments:
    del data_analysis_pipeline_arguments['experiment']

Now, assuming you have access to KFP at http://kubeflow.oss:7001, we can setup a client, initilize the pipeline and run it in the following way:

In [14]:
import kfp
KFP_ENDPOINT = "http://kubeflow.oss:7001"
kfp_client = kfp.Client(host=KFP_ENDPOINT)

In [ ]:
%run (fill)/ice-mlops-mlch/tutorials/studying/part-7/kubeflow/local_cloud_parallel_pipeline.py

In [ ]:
from icebreaker.kubeflow.use import kubeflow_manage_run
data_analysis_pipeline_time = kubeflow_manage_run(
    storage_client = workflow_setup['swift-client'],
    kfp_client = kfp_client,
    pipeline_function = local_cloud_parallel_pipeline,
    run_name = pipeline_run_name,
    experiment_name = pipeline_experiment_name,
    pipeline_arguments = data_analysis_pipeline_arguments,
    cache_steps = False,
    time_name = 'kubeflow-pipeline-local-lt1',
    time_update_object = time_object_name_update,
    loop_amount = 20000,
    loop_wait = 10
)

When the pipeline has completed, we can get the times and costs of each pipeline step across infrastructure in the following way:

In [ ]:
from icebreaker.objects.use import objects_get_data
pipeline_time_object = objects_get_data(
    swift_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'pipeline',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-name': 'time',
        'object-serialization': 'parquet',
        'path-replacers': {
            'name': 'local-cloud-workflow-1.parquet'
        },
        'path-names': [],
        'debug-prints': True,
        'lock-parameters': {},
        'lock-location': None,
        'overwrite': True
    },
    dict_format = False
)

In [13]:
%load_ext autoreload
%autoreload 2

In [16]:
from icebreaker.workflow.utility import workflow_time_stats

time_stats = workflow_time_stats(
    time_df = pipeline_time_object[0],
    cluster_yaml = pipeline_cluster_yaml
) 

In [11]:
print(f'Total pipeline cost: {time_stats['metrics']['total-pipeline-cost-eur']}')

Total pipeline cost: 7.3515464379291


In [17]:
time_stats['tables'][0]

,name,start,end,time,infrastructure,stage,cost
0,kubeflow-pipeline-local-lt1,1.793104e+09,1.793118e+09,14770.41832,local-lt1,step,0.257448
1,global-distribution-step-cloud-vm1,1.793104e+09,1.793104e+09,3.66534,cloud-vm1,step,0.001283
2,ray-external-data-analysis-cloud-vm2-step-1,1.793104e+09,1.793104e+09,14.00825,cloud-vm2,step-1,0.001879
3,ray-external-data-analysis-local-lt2-step-4,1.793104e+09,1.793104e+09,26.43968,local-lt2,step-4,0.000532
4,ray-external-data-analysis-local-lt1-step-4,1.793104e+09,1.793104e+09,48.36417,local-lt1,step-4,0.000843
5,ray-external-data-analysis-cloud-vm2-step-2,1.793104e+09,1.793104e+09,59.70090,cloud-vm2,step-2,0.008010
6,ray-external-data-analysis-local-lt3-step-3,1.793104e+09,1.793104e+09,172.32264,local-lt3,step-3,0.007894
7,ray-external-data-analysis-local-lt3-step-4,1.793104e+09,1.793104e+09,26.12182,local-lt3,step-4,0.001197
8,ray-external-data-analysis-cloud-vm2-step-3,1.793104e+09,1.793104e+09,118.75994,cloud-vm2,step-3,0.015934
9,ray-external-data-analysis-cloud-vm2-step-4,1.793104e+09,1.793104e+09,21.81638,cloud-vm2,step-4,0.002927


In [18]:
time_stats['tables'][1]

time                                                  \
                        sum         mean          std      median       min   
infrastructure                                                                
cloud-vm1       40881.43937  3716.494488  4468.361600  2358.45106   3.66534   
cloud-vm2       14150.16310  2021.451871  3472.562929   118.75994  14.00825   
local-lt1       17008.07042  3401.614084  6378.560611   407.53119  48.36417   
local-lt2        3331.77942   832.944855   945.023073   556.77609  26.43968   
local-lt3        2599.28161   519.856322   597.811540   354.57402  26.12182   

                                  cost                                \
                        max        sum      mean       std    median   
infrastructure                                                         
cloud-vm1       14309.08372  14.308504  1.300773  1.563927  0.825458   
cloud-vm2        9494.35676   1.898527  0.271218  0.465914  0.015934   
local-lt1       14770.41832   0.296451  0.059290  0.111178  0.007103   
local-lt2        2191.78756   0.067035  0.016759  0.019014  0.011202   
local-lt3        1538.43291   0.119073  0.023815  0.027386  0.016243   

                                    
                     min       max  
infrastructure                      
cloud-vm1       0.001283  5.008179  
cloud-vm2       0.001879  1.273858  
local-lt1       0.000843  0.257448  
local-lt2       0.000532  0.044099  
local-lt3       0.001197  0.070476

In [19]:
time_stats['tables'][2]

time                                                    \
                sum         mean          std      median         min   
stage                                                                   
step    46742.82894  6677.546991  6035.692295  3462.27052     3.66534   
step-1     14.00825    14.008250          NaN    14.00825    14.00825   
step-2     59.70090    59.700900          NaN    59.70090    59.70090   
step-3    899.61783   299.872610   268.647929   172.32264   118.75994   
step-4    146.54644    29.309288    10.815853    26.12182    21.81638   
step-5   6071.57655  1214.315310  1080.388143   678.76658   308.62967   
step-6   3383.23254   676.646508   494.476965   434.78560   354.57402   
step-7  20653.22247  4130.644494  3522.807800  2191.78756  1473.12707   

                          cost                                          \
                max        sum      mean       std    median       min   
stage                                                                    
step    14770.41832  11.447792  1.635399  1.808806  0.960601  0.001283   
step-1     14.00825   0.001879  0.001879       NaN  0.001879  0.001879   
step-2     59.70090   0.008010  0.008010       NaN  0.008010  0.008010   
step-3    608.53525   0.236815  0.078938  0.116159  0.015934  0.007894   
step-4     48.36417   0.013830  0.002766  0.003246  0.001197  0.000532   
step-5   2903.15659   1.017434  0.203487  0.268511  0.023264  0.005379   
step-6   1538.36428   0.465289  0.093058  0.113058  0.016243  0.007103   
step-7   9494.35676   3.498540  0.699708  0.939040  0.070476  0.025677   

                  
             max  
stage             
step    5.008179  
step-1  0.001879  
step-2  0.008010  
step-3  0.212987  
step-4  0.008332  
step-5  0.585618  
step-6  0.226792  
step-7  2.084431

In [20]:
time_stats['tables'][3]

time                                          \
                               sum          mean          std       median   
infrastructure stage                                                         
cloud-vm1      step    31972.41062   5328.735103  5332.300844   3103.42264   
               step-3    608.53525    608.535250          NaN    608.53525   
               step-4     23.80439     23.804390          NaN     23.80439   
               step-5   1673.19349   1673.193490          NaN   1673.19349   
               step-6    647.97745    647.977450          NaN    647.97745   
               step-7   5955.51817   5955.518170          NaN   5955.51817   
cloud-vm2      step-1     14.00825     14.008250          NaN     14.00825   
               step-2     59.70090     59.700900          NaN     59.70090   
               step-3    118.75994    118.759940          NaN    118.75994   
               step-4     21.81638     21.816380          NaN     21.81638   
               step-5   2903.15659   2903.156590          NaN   2903.15659   
               step-6   1538.36428   1538.364280          NaN   1538.36428   
               step-7   9494.35676   9494.356760          NaN   9494.35676   
local-lt1      step    14770.41832  14770.418320          NaN  14770.41832   
               step-4     48.36417     48.364170          NaN     48.36417   
               step-5    308.62967    308.629670          NaN    308.62967   
               step-6    407.53119    407.531190          NaN    407.53119   
               step-7   1473.12707   1473.127070          NaN   1473.12707   
local-lt2      step-4     26.43968     26.439680          NaN     26.43968   
               step-5    678.76658    678.766580          NaN    678.76658   
               step-6    434.78560    434.785600          NaN    434.78560   
               step-7   2191.78756   2191.787560          NaN   2191.78756   
local-lt3      step-3    172.32264    172.322640          NaN    172.32264   
               step-4     26.12182     26.121820          NaN     26.12182   
               step-5    507.83022    507.830220          NaN    507.83022   
               step-6    354.57402    354.574020          NaN    354.57402   
               step-7   1538.43291   1538.432910          NaN   1538.43291   

                                                      cost            \
                               min          max        sum      mean   
infrastructure stage                                                   
cloud-vm1      step        3.66534  14309.08372  11.190344  1.865057   
               step-3    608.53525    608.53525   0.212987  0.212987   
               step-4     23.80439     23.80439   0.008332  0.008332   
               step-5   1673.19349   1673.19349   0.585618  0.585618   
               step-6    647.97745    647.97745   0.226792  0.226792   
               step-7   5955.51817   5955.51817   2.084431  2.084431   
cloud-vm2      step-1     14.00825     14.00825   0.001879  0.001879   
               step-2     59.70090     59.70090   0.008010  0.008010   
               step-3    118.75994    118.75994   0.015934  0.015934   
               step-4     21.81638     21.81638   0.002927  0.002927   
               step-5   2903.15659   2903.15659   0.389517  0.389517   
               step-6   1538.36428   1538.36428   0.206402  0.206402   
               step-7   9494.35676   9494.35676   1.273858  1.273858   
local-lt1      step    14770.41832  14770.41832   0.257448  0.257448   
               step-4     48.36417     48.36417   0.000843  0.000843   
               step-5    308.62967    308.62967   0.005379  0.005379   
               step-6    407.53119    407.53119   0.007103  0.007103   
               step-7   1473.12707   1473.12707   0.025677  0.025677   
local-lt2      step-4     26.43968     26.43968   0.000532  0.000532   
               step-5    678.76658    678.76658   0.013657  0.013657   
               step-6    434.7856

With these, we can start processing the collected results using http://mlflow.oss:7001, clicking 'mlch-code-assistant-rag-pipeline' experiment and comparing the various results. We will take the first 3 datasets when the are multiple samples. We can gather the following details from the used material and processing times:

- Local internal material:
    - All chunks are in english
    - There are around 84+ markdown chunks
    - There are around 19+ jupyter chunks
    - There are around 83+ python chunks
    - There are around 5+ bash chunks
    - There are around 159+ yaml chunks
    - There are around 95+ text chunks
    - There are around 2+ env chunks
    - Mean rows is between 26-47
    - Mean charactes is between 988-1888
    - There are around 103+ relevance 3 chunks
    - There are around 222+ relevance 2 chunks
    - There are around 122+ relevance 1 chunks
    - There are 447+ chunks
    - Processing time is around 1 min
      
- Local external material:
    - All QA pairs are in english
    - Human eval has 164 python pairs
    - Mbpp has 427 python pairs
    - Bash command has 6153 bash pairs
    - Mean question rows is between 1-14
    - Mean question characters is 81-450
    - Mean answer rows is between 1.4-6.7
    - Mean answer characters is 63-180
    - Processing time is 10 sec
      
- Local-cloud data analysis pipeline:
    - Total pipeline time is 14770.41832 sec (246 min 10 sec)
    - Step-1 processing time is 14.00825 sec
    - Step-2 processing time is 59.7 sec
    - Step-3 processing time is 608.53525 sec (10 min 9 sec)
    - Step-4 processing time is 48.36417 sec 
    - Step-5 processing time is 2903.15659 sec (48 min 23 sec)
    - Step-6 processing time is 1538.36428 sec (25 min 38 sec)
    - Step-7 processing time is 9494.35676 sec (158 min 14 sec)
    - Total pipeline cost is 7.35 EUR
      
- Local-cloud external material:- 
    - educational-instruct (step-1):
        - fully python and english
        - 118278 python pairs
        - 3 chinese and 1 spanish false positives
        - Mean question rows is between 1.27
        - Mean question characters is 163
        - Mean answer rows is between 14.09
        - Mean answer characters is 362
        - Processing time is 14.00825 sec
    - mceval-instruct (step-2):
        - Mixed formats and majority english
        - 2 de, 1 es, 2 fr, 1 pt, 1 tr and 25 zh cases
        - 3840 python pairs
        - 5 markdown pairs
        - 12 text pairs
        - 903 bash pairs and 221 sh pairs
        - 1 yaml pair
        - Mean question rows is between 13.90
        - Mean question characters is 1074
        - Mean answer rows is between 77.69
        - Mean answer characters is 2607
        - Processing time is 59.7 sec
    - evol-instruct (step-3):
        - mixed formats and majority english
        - instruct-1:
            - 18321 python pairs
            - 959 markdown pairs
            - 46 yaml pairs
            - 1 text pair
            - 437 bash pair and 28 sh pairs
            - Mean question rows is between 9.49
            - Mean question characters is 641
            - Mean answer rows is between 31.96
            - Mean answer characters is 1548
        - instruct-2:
            - 1 ar, 1 fi, 5 de, 1 et, 1 fr, 2 ja, 2 no, 1, ru, 1 vi , 1 it, 1 la, 2 pt, 2 sv and 146 zh cases
            - 18231 python pairs
            - 974 markdown pairs
            - 33 yaml pairs
            - 1 text pair
            - 454 bash pair and 35 sh pairs
            - Mean question rows is between 10.03
            - Mean question characters is 664
            - Mean answer rows is between 32.05
            - Mean answer characters is 1552
        - instruct-3:
            - 1 fi, 2 fr, 2 la, 1 pt, 1 ru, 1 ta, 1 vi, 2 de, and 120 zh cases
            - 14446 python pairs
            - 726 markdown pairs
            - 17 yaml pairs
            - 386 bash pairs and 26 sh pairs
            - Mean question rows is between 9.64
            - Mean question characters is 650
            - Mean answer rows is between 32.26
            - Mean answer characters is 1557
        - Processing time is 608.53525 sec
    - package-instruct (step-4):
        - mostly python and english
        - instruct-1:
            - 29981 python pairs
            - 4 markdown pairs (false positive)
            - 5 bash pairs
            - 1 de, 1 es, 144 zh, ru 1 and 1 zh
            - Mean question rows is between 35.6
            - Mean question characters is 1644
            - Mean answer rows is between 50
            - Mean answer characters is 2118
        - instruct-2:
            - 29980 python pairs
            - 2 markdown pairs (false positives)
            - 4 bash pairs
            - 1 txt pair (false positives)
            - 1 pt and 94 zh
            - Mean question rows is between 28
            - Mean question characters is 1327
            - Mean answer rows is between 43
            - Mean answer characters is 1685 
        - instruct-3:
            - 29984 python pairs and 1 py pair
            - 4 bash pairs
            - 86 zh
            - Mean question rows is between 28
            - Mean question characters is 1327
            - Mean answer rows is between 43
            - Mean answer characters is 1685
        - Processing time is 48.36417 sec
    - filtered-infinity-instruct (step-5):
        - mixed format and majority english
        - instruct-1 (lt2):
            - 4887 zh, 39 ru, 21 es, 10 fr, 8 ko and others
            - 29375 python pairs
            - 1076 markdown pairs
            - 392 bash pairs and 25 sh pairs
            - 11411 txt pairs
            - 64 yaml pairs
            - Mean question rows is between 9.7
            - Mean question characters is 588
            - Mean answer rows is between 27
            - Mean answer characters is 1239
        - instruct-2 (vm2):
            - 4827 zh, 30 ru, 21 es, 16 fr and 13 pt among others
            - python 29386 pairs
            - 1030 markdown pairs
            - 339 bash pairs and 17 sh
            - 78 yaml pairs
            - Mean question rows is between 9.8
            - Mean question characters is 593
            - Mean answer rows is between 27
            - Mean answer characters is 1235
        - instruct-3 (vm2):
            - 2813 zh, 21 es, 42 ru and de 8 among others
            - 29480 python pairs
            - 386 bash pairs and 11 sh pairs
            - 1044 markdown pairs
            - 63 yaml pairs
            - 1 text pair
            - Mean question rows is between 9.8
            - Mean question characters is 590
            - Mean answer rows is between 27
            - Mean answer characters is 1235
        - Processing time is 2903.15659 sec (48 min 23 sec)
    - realuser-instruct (step-6):
        -  Mixed in format and small majority english
        -  Instruct-1 (vm2):
            - 15417 en, 14566 zh, 10 ru, 3 fr among others
            - 12542 python pairs
            - 1030 markdown pairs
            - 1350 bash pairs and 944 sh pairs
            - 1 text pair
            - 69 yaml pair
            - Mean question rows is between 11.77
            - Mean question characters is 637
            - Mean answer rows is between 58
            - Mean answer characters is 2097
        -  Instruct-2 (lt2):
            - 15423 en, 14739 zh, 9 ru and 2 es among others
            - 12618 python pairs
            - 1290 bash pairs abd 1015 sh pairs
            - 1058 markdown pairs
            - 53 yaml pairs
            - Mean question rows is between 11.87
            - Mean question characters is 638
            - Mean answer rows is between 58
            - Mean answer characters is 2106
        -  Instruct-3 (vm1):
            - 15246 en, 14732 zh, 8 ru, 3 es and 2 tr among others
            - 12401 python pairs
            - 1381 bash pairs and 959 sh pairs
            - 1105 markdown pairs
            - 67 yaml pairs
            - Mean question rows is between 11.71
            - Mean question characters is 643
            - Mean answer rows is between 58
            - Mean answer characters is 2099
        - Processing time is 1538.36428 sec (25 min 38 sec)
    - largescale-diverse-instruct (step-7):
        - mixed in format and small majority english
        - Instruct-1 (lt3):
            - 23693 en and 16307 zh
            - 14685 python pairs
            - 798 bash pairs and 65 sh
            - 3774 markdown
            - 33 yaml
            - Mean question rows is between 7.94
            - Mean question characters is 447
            - Mean answer rows is between 34
            - Mean answer characters is 1598
        - Instruct-2 (vm2):
            - 23860 en and 16140 zh
            - 14877 python pairs
            - 742 bash pairs and 225 sh pairs
            - 3681 markdown pairs
            - 31 yaml pairs
            - Mean question rows is between 7.95
            - Mean question characters is 449
            - Mean answer rows is between 34
            - Mean answer characters is 1598
        - Instruct-3 (lt1):
            - 23734 en, 16264 zh, 2 es and among others
            - 14729 python pairs
            - 772 bash pairs and 244 sh pairs
            - 3791 markdown pairs
            - 31 yaml pairs
            - Mean question rows is between 7.98
            - Mean question characters is 448
            - Mean answer rows is between 34
            - Mean answer characters is 1604
        - Processing time is 9494.35676 sec (158 min 14 sec)

With these details, we can conclude the following actions for the preprocessing stage:

- Internal dataset actions:
    - Our validation dataset will have around 150 pairs using grade 3 materials
    - Chunks with grade 2 could be used for multi document reasoning
    - The validation dataset would consist mostly of markdown/jupyter questions that mostly have python with occational yaml and bash
    - For the synthetic dataset, we will use the internal material to produce the following types:
        - Factual, which is meant to check RAG performance
        - Synthesis, which is meant to check models multi document reasoning
        - Negative, which meant to check application guardrails 
          
- External dataset actions:̈́
    - The datasets have the following query formats:
        - Bash data asks models to produce a Bash command for the question
        - Human eval ask models to fill a Python function based on comments inside
        - Mbpp asks models to produce a Python function base on question
        - Package instructs models to use context to produce proper Python code with explanation
        - Educational instructs models to pporduce python functions based on quesiton
        - Mceval asks models with proper context to prodcue proper code cases with small markdwon
        - Largescale ask model with indepth context to produce proper coding solution
        - Evol asks models with general user context to produce different coding solutions
        - Realuser ask model with general user context to produce an general answer
        - Filtered ask model with general user context to produce an genearal answer
    - We can select various python, bash, markdown and yaml to form a general datasets that evalutes assistants general abilities
    - We can sample the wanted types with the following order
        1. Bash command for simple bash queries
        2. Humaneval, mbpp and educational for simple python queries
        3. Package for fully explained general python queries
        4. Mceval and largescale for contextually limited complex python, bash, markdown and yaml queries
        5. Evol, real user and filtered for user context limited ocmplex python, bash, markdown and yaml queries

- General workflow actions:
    - The varied amount of characters between 81-2200 means that LLMs need to have atleast 16k context
    - The RAG might require some formatting to fit everything into the context
    - The general and synthetic datasets can only give direction, which is why validation dataset and human evalution are primary source of truth

We now have done the necessery data analysis to proceed to data preprocessing stage.

---